In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
! pip install -r '/content/drive/MyDrive/Master/requierments.txt'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 20.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.4/533.4 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 148.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 156.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 1

In [ ]:
! pip install --upgrade "mistral-common[audio]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 133.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 153.0 MB/s eta 0:00:00


In [ ]:
from transformers import VoxtralForConditionalGeneration, AutoProcessor
import torch
from peft import PeftModel
import pandas as pd
from tqdm import tqdm
import random
import re

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="mistralai/Voxtral-Mini-3B-2507", local_dir="/voxtral-mini")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

'/voxtral-mini'

In [ ]:
device = "cuda"
model_id = "mistralai/Voxtral-Mini-3B-2507" # or folder path
processor = AutoProcessor.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/mistralai/Voxtral-Mini-3B-2507/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/mistralai/Voxtral-Mini-3B-2507/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/mistralai/Voxtral-Mini-3B-2507/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].

preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

HTTP Error 503 thrown while requesting HEAD https://huggingface.co/mistralai/Voxtral-Mini-3B-2507/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


config.json: 0.00B [00:00, ?B/s]

tekken.json:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

In [ ]:
model = VoxtralForConditionalGeneration.from_pretrained(model_id,
                                                             dtype=torch.bfloat16,
                                                             device_map=device)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/108 [00:00<?, ?B/s]

In [ ]:
base_model = VoxtralForConditionalGeneration.from_pretrained(model_id,
                                                             dtype=torch.bfloat16,
                                                             device_map=device)
model = PeftModel.from_pretrained(base_model, "/content/Voxtral-fine-tune")

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Master/data/test.csv')
df.head()

,Unnamed: 0,artist,title,Wonder,Transcendence,Nostalgia,Tenderness,Peacefulness,Joy,Power,Tension,Sadness,Spotify ID,Youtube ID,genre,Path
0,457,Sugababes,Push The Button,5,2,4,2,2,5,5,3,1,3EDS89HeAhHiZKlljciK0a,Ugkxu8HzYufqmVieUjWOKqkoQqrrFpFbRnh7,pop,tracks/Ugkxu8HzYufqmVieUjWOKqkoQqrrFpFbRnh7.mp3
1,148,Lacrimas Profundere,The Crown of Leaving,3,3,3,2,3,2,3,4,3,7qNpAPSfPiLQ47cez2IpeW,NaN,rock,tracks/7qNpAPSfPiLQ47cez2IpeW.mp3
2,742,The Cinematic Orchestra,Time and Space,3,3,2,3,5,1,1,3,5,6qqNMdHr1hcyF4amMDP5Sf,NaN,electronic,tracks/6qqNMdHr1hcyF4amMDP5Sf.mp3
3,526,The Beach Boys,Good Vibrations,4,2,4,2,4,5,3,1,1,6aU6a9tdn2vHhnPGlboFZX,UgkxoWtHfVqF7dqNB3LH94zBUkV0DScEs0Gm,pop,tracks/UgkxoWtHfVqF7dqNB3LH94zBUkV0DScEs0Gm.mp3
4,400,Joshua Kadison,Jessie,4,3,4,5,4,2,3,2,4,4n8iSiSWRdaSeSpJMbdk9O,NaN,pop,tracks/4n8iSiSWRdaSeSpJMbdk9O.mp3


# Randomized Emotions Order Prediction

In [ ]:
emo = ['Wonder (Filled with wonder, Dazzled, Allured, Moved)',
       'Transcendence (Fascinated, Overwhelmed, Feelings of transcendence and spirituality)',
       'Nostalgia (Nostalgic, Dreamy, Sentimental, Melancholic)',
       'Tenderness (Tender, Affectionate, In love, Mellowed)',
       'Peacefulness (Serene, Calm, Soothed, Relaxed)',
       'Joy (Joyful, Amused, Animated, Bouncy)',
       'Sadness (Sad, Sorrowful)',
       'Power (Strong, Triumphant, Energetic, Fiery)',
       'Tension (Tense, Agitated, Nervous, Irritated)']

In [ ]:
for i, row in tqdm(df.iterrows(), total=115):

    # Randomize emotions list
    random.shuffle(emo) # <- Comment the line for Un-randomized predictions
    conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text",
                     "text":f"""Rate the intensity of the following emotions induced by this music excerpt.
Use the following scale : 1 (not at all), 2 (Somewhat), 3 (Moderately), 4 (Quite a lot), 5 (Very Much).
Just give the ratings, without justifying.
- {emo[0]}
- {emo[1]}
- {emo[2]}
- {emo[3]}
- {emo[4]}
- {emo[5]}
- {emo[6]}
- {emo[7]}
- {emo[8]}"""         },
                    {"type": "audio",
                     "path": f'/content/drive/MyDrive/Master/{row['Path']}'},
                ],
            },
        ]

    inputs = processor.apply_chat_template(
        conversation,
        )
    inputs = inputs.to(device, dtype=torch.bfloat16)

    outputs = model.generate(**inputs, max_new_tokens=1024)
    decoded_outputs = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    with open("/content/drive/MyDrive/Master/Voxtral_randomized_results.txt", "a+", encoding="utf-8") as f:
        f.write("=" * 20 + '\n' + str(decoded_outputs[0]) + '\n')

100%|██████████| 115/115 [08:41<00:00,  4.53s/it]


# Single Emotion Prediction

In [ ]:
for e in emo :

  pattern = r'^\w+'
  single_emo = re.findall(pattern, e)[0]
  print(f'Prediction for {single_emo} :')

  for i, row in tqdm(df.iterrows(), total=115):

      conversation = [
              {
                  "role": "user",
                  "content": [
                      {"type": "text",
                       "text":f"""Rate the intensity of {e} induced by this music excerpt.
  Use the following scale : 1 (not at all), 2 (Somewhat), 3 (Moderately), 4 (Quite a lot), 5 (Very Much).
  Just give the numeric rating, without justifying.
  {e}
  """         },
                      {"type": "audio",
                       "path": f'/content/drive/MyDrive/Master/{row['Path']}'},
                  ],
              },
          ]

      inputs = processor.apply_chat_template(
        conversation,
        )
      inputs = inputs.to(device, dtype=torch.bfloat16)

      outputs = model.generate(**inputs, max_new_tokens=1024)
      decoded_outputs = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

      with open(f"/content/drive/MyDrive/Master/Voxtral_{single_emo}_results.txt", "a+", encoding="utf-8") as f:
          f.write(str(decoded_outputs) + '\n')

Prediction for Joy :


100%|██████████| 115/115 [01:57<00:00,  1.03s/it]


Prediction for Transcendence :


100%|██████████| 115/115 [01:53<00:00,  1.01it/s]


Prediction for Tension :


...

100%|██████████| 115/115 [03:00<00:00,  1.57s/it]


Prediction for Sadness :


100%|██████████| 115/115 [01:51<00:00,  1.03it/s]


Prediction for Wonder :


100%|██████████| 115/115 [01:53<00:00,  1.01it/s]


# Prediction for the original data score

In [ ]:
for i, row in tqdm(df.iterrows(), total=115):

    # Randomize emotions list
    # random.shuffle(emo) # <- Comment the line for Un-randomized predictions
    conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text",
                     "text":f"""Rate the intensity of the following emotions induced by this music excerpt, using a 0-100 scale.
Just give the numeric ratings, without justifying.
- {emo[0]}
- {emo[1]}
- {emo[2]}
- {emo[3]}
- {emo[4]}
- {emo[5]}
- {emo[6]}
- {emo[7]}
- {emo[8]}"""         },
                    {"type": "audio",
                     "path": f'/content/drive/MyDrive/Master/{row['Path']}'},
                ],
            },
        ]

    inputs = processor.apply_chat_template(
        conversation,
        )
    inputs = inputs.to(device, dtype=torch.bfloat16)

    outputs = model.generate(**inputs, max_new_tokens=1024)
    decoded_outputs = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    with open("/content/drive/MyDrive/Master/Voxtral_og_results.txt", "a+", encoding="utf-8") as f:
        f.write("=" * 20 + '\n' + str(decoded_outputs[0]) + '\n')

100%|██████████| 115/115 [10:01<00:00,  5.23s/it]
